# Session 06 · Analysis of Variance

Run each cell with **Shift+Enter**. Original material by Aseef Ahmed.

## Example 1 - why not many t-tests?

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
for k in (1, 3, 6, 10):
    print(f"{k:>2} tests -> P(at least one false positive) = {1 - 0.95 ** k:.3f}")

 1 tests -> P(at least one false positive) = 0.050
 3 tests -> P(at least one false positive) = 0.143
 6 tests -> P(at least one false positive) = 0.265
10 tests -> P(at least one false positive) = 0.401


## Data: four teaching methods (synthetic)

In [2]:
rng = np.random.default_rng(6)
effects = {"lecture": 62, "flipped": 68, "project": 71, "self_paced": 63}
df = pd.DataFrame([(m, rng.normal(mu, 9)) for m, mu in effects.items() for _ in range(30)], columns=["method", "score"])
k, N = df["method"].nunique(), len(df)
print(df.groupby("method")["score"].agg(["count", "mean", "std"]).round(1))

            count  mean   std
method                       
flipped        30  68.6   7.7
lecture        30  63.7  10.0
project        30  69.8   7.8
self_paced     30  60.1  10.5


## Example 2 - ANOVA table by hand

In [3]:
grand = df["score"].mean()
ssb = df.groupby("method")["score"].apply(lambda g: len(g) * (g.mean() - grand) ** 2).sum()
ssw = df.groupby("method")["score"].apply(lambda g: ((g - g.mean()) ** 2).sum()).sum()
F = (ssb / (k - 1)) / (ssw / (N - k)); p = stats.f.sf(F, k - 1, N - k)
print(f"SSB {ssb:.1f}  SSW {ssw:.1f}  F {F:.2f}  p {p:.2e}")

SSB 1805.7  SSW 9582.3  F 7.29  p 1.61e-04


## Example 3 - scipy and statsmodels

In [4]:
groups = [g["score"] for _, g in df.groupby("method")]
print(stats.f_oneway(*groups))
table = sm.stats.anova_lm(smf.ols("score ~ C(method)", data=df).fit(), typ=2)
print(table)
assert np.isclose(table.loc["C(method)", "F"], F)

F_onewayResult(statistic=np.float64(7.286484754810699), pvalue=np.float64(0.00016094318756874705))
                sum_sq     df         F    PR(>F)
C(method)  1805.724626    3.0  7.286485  0.000161
Residual   9582.309519  116.0       NaN       NaN


## Example 4 - assumptions

In [5]:
print("Levene p:", round(stats.levene(*groups).pvalue, 3))
print("Shapiro p (residuals):", round(stats.shapiro(smf.ols("score ~ C(method)", df).fit().resid).pvalue, 3))

Levene p: 0.184
Shapiro p (residuals): 0.708


## Example 5 - Tukey HSD and eta squared

In [6]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd
print(pairwise_tukeyhsd(df["score"], df["method"], alpha=0.05))
print("eta squared:", round(ssb / (ssb + ssw), 3))

   Multiple Comparison of Means - Tukey HSD, FWER=0.05    
 group1   group2   meandiff p-adj   lower    upper  reject
----------------------------------------------------------
flipped    lecture  -4.9337 0.1584 -11.0508  1.1834  False
flipped    project   1.1576 0.9605  -4.9595  7.2747  False
flipped self_paced   -8.494 0.0024 -14.6111 -2.3769   True
lecture    project   6.0913 0.0514  -0.0258 12.2084  False
lecture self_paced  -3.5603 0.4306  -9.6774  2.5568  False
project self_paced  -9.6516 0.0004 -15.7687 -3.5345   True
----------------------------------------------------------
eta squared: 0.159


## Example 6 - two-way ANOVA with interaction

In [7]:
rows = []
for fert, f_eff in {"urea": 0, "DAP": 4, "organic": 2}.items():
    for irr, i_eff in {"canal": 0, "drip": 5}.items():
        bonus = 4 if (fert == "DAP" and irr == "drip") else 0
        rows += [(fert, irr, 40 + f_eff + i_eff + bonus + rng.normal(0, 3)) for _ in range(12)]
farm = pd.DataFrame(rows, columns=["fertiliser", "irrigation", "yield_kg"])
print(sm.stats.anova_lm(smf.ols("yield_kg ~ C(fertiliser) * C(irrigation)", data=farm).fit(), typ=2).round(4))

                               sum_sq    df        F  PR(>F)
C(fertiliser)                273.6841   2.0  17.1103  0.0000
C(irrigation)                731.9412   1.0  91.5195  0.0000
C(fertiliser):C(irrigation)    7.8561   2.0   0.4911  0.6141
Residual                     527.8452  66.0      NaN     NaN
